<h1 style="font-size: 36px; color: blue;">STOWA Proevenverzameling tool v5.0</h1>

## Benodigde installaties

In [ ]:
# Importeren van benodigde package
# Na het eerte keer installeren van de benodigde packages de kernel handmatig opnieuw optarten via het menu Kernel → Restart

path_to_wheel = r"C:\Users\deenekat7271\Documents\GitHub\pv-tool\dist\pv_tool-0.2.7-py3-none-any.whl"
!pip install "{path_to_wheel}"

In [ ]:
# all imports
import importlib.util
import ipywidgets as widgets
from IPython.display import display, Markdown
from ipyfilechooser import FileChooser
from pathlib import Path
import os

from pv_tool.imports.import_data import Dbase
from pv_tool.cphi_analysis.c_phi_analysis import CPhiAnalyse
from pv_tool.cphi_analysis.variables import *

# widgetfuncties nog verder aanvullen!
from widgetfunctie_cphi import *

In [ ]:
# Controleert en installeer indien nodig
packages = ['openpyxl', 'ipyfilechooser', 'pandas_schema', 'xlsxwriter', 'ipywidgets', 'reportlab', 'reportlab', 'kaleido']
for package in packages:
    check_package_install(package)

## Stap 1: Inladen van benodigde data

In [ ]:
# Data inladen
dropdown_template = create_import_dropdown()
file_chooser_import = create_file_chooser()
display(Markdown("**Stap 1: Kies Excel template voor uploaden data:**"))
display(dropdown_template)
display(file_chooser_import)

## Selecteer export locatie

In [ ]:
dir_chooser, name_box = select_export_location_and_name()

## Stap 2a: Importeren en valideren data (inclusief toevoegen/herberekenen analyse kolommen)

In [ ]:
import time

In [ ]:
timestart = time.time()
dbase = Dbase()

if not dropdown_template.value:
    print("Er is geen template geselecteerd.")
elif not file_chooser_import.selected:
    print("Er is geen bestand geselecteerd. Selecteer een bestand voordat je verder gaat.")
elif not dir_chooser.selected_path:
    print("Selecteer een exportlocatie.")
else:
    chosen_filename = name_box.value if name_box.value else None
    process_import_and_validate(
        dbase=dbase,
        template_name=dropdown_template.value,
        file_path=file_chooser_import.selected,
        export_dir=Path(dir_chooser.selected_path)
    )
print(f"Duur: {time.time() - timestart:.2f} seconden")

## Stap 2b: Als errors zijn opgelost exporteer naar Template

In [ ]:
# Export dbase-template
timestart = time.time()
process_export(
    dbase,
    export_dir=Path(dir_chooser.selected_path),
    filename=name_box.value if name_box.value else None
)
print(f"Duur: {time.time() - timestart:.2f} seconden")

## Bepalen gedraineerde parameters triaxiaalproeven en DSS-proeven op basis van fit (C en Phi)

In [ ]:
# Dataframe met proefdata
dbase_df = dbase.dbase_df

# Lijsten maken
PV_txt_lijst, PV_dss_lijst = maak_verzamelings_lijsten(dbase_df)

# Widgets aanmaken
(dropdown_type_proef, dropdown_rekpercentage_txt, dropdown_rekpercentage_dss,
 dropdown_verzameling, multi_select_verzameling, container_rekpercentage, output_rekpercentage) = maak_proef_widgets(PV_txt_lijst, PV_dss_lijst)

# 'gekozen_rekpercentage' als lijst zodat het binnen callbacks aanpasbaar blijft
gekozen_rekpercentage = ['eindsterkte']

# Koppel callbacks
koppel_callbacks(
    dropdown_type_proef, dropdown_rekpercentage_txt, dropdown_rekpercentage_dss,
    dropdown_verzameling, multi_select_verzameling, container_rekpercentage, output_rekpercentage,
    PV_txt_lijst, PV_dss_lijst, gekozen_rekpercentage
)

# Toon alles
toon_widgets(
    dropdown_type_proef, dropdown_verzameling, container_rekpercentage, output_rekpercentage, multi_select_verzameling
)

In [ ]:
analyse, coh_gem, phi_kar, coh_kar, partphi, partcoh, typeverzameling = voer_cphi_analyse_uit(
    dbase,
    dropdown_verzameling,
    dropdown_type_proef,
    dropdown_rekpercentage_txt,
    dropdown_rekpercentage_dss,
    dir_chooser,      # FileChooser uit select_export_location_and_name
    name_box,         # Text widget uit select_export_location_and_name
    gekozen_rekpercentage,
    toon_cphi_tabel
)